# VeloFetch — Google Colab Runner

Run the Python-only VeloFetch Telegram bot from a Colab runtime. The notebook installs FFmpeg, aria2 and Python dependencies, securely prompts for Telegram credentials, then starts the bot with live logs.

**Keep this notebook running** while you want the bot online. Colab runtimes are temporary; the in-memory queue and local downloads are cleared when the runtime is deleted or reset.

In [ ]:
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/Telebotfaroff/Content-india.git"
PROJECT_DIR = Path("/content/Content-india")

if not (PROJECT_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True)

%cd /content/Content-india

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg aria2
!ffmpeg -version | head -n 1
!aria2c --version | head -n 1

In [ ]:
!python -m pip install --upgrade pip
!python -m pip install -r requirements.txt

## Telegram credentials

Credentials are requested at runtime and kept in this Colab session's environment only. They are not written into the notebook or repository.

In [ ]:
import os
from getpass import getpass

os.environ["BOT_TOKEN"] = getpass("Telegram bot token: ").strip()
os.environ["API_ID"] = input("Telegram API ID: ").strip()
os.environ["API_HASH"] = getpass("Telegram API hash: ").strip()

target_channel = input("Target channel ID or @username (optional, press Enter to skip): ").strip()
if target_channel:
    os.environ["TARGET_CHANNEL"] = target_channel
else:
    os.environ.pop("TARGET_CHANNEL", None)

# Colab-friendly defaults; change these values if needed.
os.environ["DOWNLOAD_WORKERS"] = "1"
os.environ["UPLOAD_WORKERS"] = "1"
os.environ["MAX_PENDING_UPLOADS"] = "1"
os.environ["PROGRESS_UPDATE_INTERVAL"] = "3"

if not os.environ["BOT_TOKEN"] or not os.environ["API_ID"] or not os.environ["API_HASH"]:
    raise ValueError("BOT_TOKEN, API_ID and API_HASH are required.")
print("Credentials loaded into this runtime.")

## Start VeloFetch

Run this cell to start the bot. Logs will appear below. Stop the cell to shut down the bot; stopping or resetting the Colab runtime also stops it.

In [ ]:
!python -u main.py